[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/14-deployment-and-llmops/notebooks/01-fastapi-from-zero.ipynb)

# 01 · FastAPI from zero

**What you will learn**

1. What an API, a route and a status code are — by building them.
2. Path parameters, query parameters and request bodies checked by **pydantic**.
3. Returning errors on purpose with `HTTPException`.
4. Why `async` matters for LLM apps — measured, not just claimed.
5. Testing an API with **TestClient**, and finally running a real server with **uvicorn**.

Everything runs offline. No API key is needed: where a real LLM call would go we use a clearly-labelled stand-in.

**Words used below**
- **API** — a fixed menu of requests one program can make to another.
- **HTTP** — the rules browsers and servers use to talk. A request has a *method* (GET = read, POST = send data) and a *path* like `/chat`.
- **Status code** — a 3-digit result: 2xx = OK, 4xx = the caller made a mistake, 5xx = the server failed.
- **JSON** — text that looks like a Python dict, e.g. `{"message": "hi"}`.

## Step 0 — check the tools

This cell installs FastAPI and friends **only if they are missing** (useful in Colab) and prints the versions. This notebook was written against fastapi 0.165, pydantic 2.13 and httpx 0.28.

In [1]:
import importlib, subprocess, sys
# Install any missing package (only matters in Google Colab; locally they are already installed).
for pkg in ["fastapi", "httpx", "pydantic", "uvicorn"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import fastapi, pydantic, httpx
print("fastapi", fastapi.__version__, "| pydantic", pydantic.__version__, "| httpx", httpx.__version__)

fastapi 0.165.0 | pydantic 2.13.5 | httpx 0.28.1


## Step 1 — the smallest possible API

- `FastAPI()` creates the **application object**.
- `@app.get("/health")` is a **decorator**: a label on the next function that says "run this when someone sends GET /health".
- Returning a Python dict is enough — FastAPI converts it to JSON.

We do not start a server yet. **TestClient** lets us send fake HTTP requests straight to the app inside this notebook.

In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Hello API")

@app.get("/health")
def health():
    return {"status": "ok"}

client = TestClient(app)            # a pretend browser that talks to `app` directly
r = client.get("/health")           # send GET /health
print("status code:", r.status_code)
print("JSON body  :", r.json())

status code: 200
JSON body  : {'status': 'ok'}


## Step 2 — a path that does not exist

Ask for a path we never defined. FastAPI answers **404 Not Found** by itself. Recognising status codes quickly is a core debugging skill.

In [3]:
r = client.get("/does-not-exist")
print(r.status_code, r.json())

404 {'detail': 'Not Found'}


## Step 3 — path parameters and query parameters

- **Path parameter**: part of the URL itself, written `{name}` in the route: `/users/42`.
- **Query parameter**: after a `?`: `/search?q=docker&limit=3`.

The **type hints** (`user_id: int`) are not decoration: FastAPI converts the text `"42"` into the number `42`, and if it cannot (`/users/abc`) it replies **422** with an explanation — your function never runs.

In [4]:
@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id, "type": str(type(user_id))}

@app.get("/search")
def search(q: str, limit: int = 10):     # q is required, limit has a default
    return {"q": q, "limit": limit}

print(client.get("/users/42").json())
print(client.get("/search", params={"q": "docker", "limit": 3}).json())
print(client.get("/search", params={"q": "docker"}).json())
bad = client.get("/users/abc")
print(bad.status_code, bad.json()["detail"][0]["msg"])

{'id': 42, 'type': "<class 'int'>"}
{'q': 'docker', 'limit': 3}
{'q': 'docker', 'limit': 10}
422 Input should be a valid integer, unable to parse string as an integer


## Step 4 — request bodies with pydantic models

For POST requests the client sends a JSON **body**. We describe its shape with a **pydantic model**: a class listing each field, its type, and rules.

- `Field(256, ge=1, le=1024)` → default 256, must be **g**reater-or-**e**qual 1 and **l**ess-or-**e**qual 1024.
- `response_model=ChatResponse` makes FastAPI check *and document* what we send back.

Our "model" is a stand-in function so the notebook needs no key. Later cells show where a real call goes.

In [5]:
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = Field(256, ge=1, le=1024)

class ChatResponse(BaseModel):
    reply: str
    input_tokens: int
    output_tokens: int

def fake_llm(prompt: str) -> str:
    """Offline stand-in for an LLM: predictable, free, instant."""
    return f"(offline demo model) You asked about: {prompt[:80]}"

def rough_tokens(text: str) -> int:
    return max(1, len(text) // 4)          # ~4 characters per token for English

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    reply = fake_llm(req.message)
    return ChatResponse(reply=reply, input_tokens=rough_tokens(req.message), output_tokens=rough_tokens(reply))

r = client.post("/chat", json={"message": "What is Docker?"})
print(r.status_code, r.json())

200 {'reply': '(offline demo model) You asked about: What is Docker?', 'input_tokens': 3, 'output_tokens': 13}


## Step 5 — watch validation reject bad data

Three bad requests: a missing field, a wrong type, and a number out of range. Each gets **422** and a precise message saying *which* field is wrong. This is free input validation — the first guardrail of any API.

In [6]:
bad_requests = [
    {},                                         # message missing
    {"message": "hi", "max_tokens": "lots"},    # wrong type
    {"message": "hi", "max_tokens": 5000},      # out of range
]
for body in bad_requests:
    r = client.post("/chat", json=body)
    err = r.json()["detail"][0]
    print(r.status_code, "| field:", err["loc"], "|", err["msg"])

422 | field: ['body', 'message'] | Field required
422 | field: ['body', 'max_tokens'] | Input should be a valid integer, unable to parse string as an integer
422 | field: ['body', 'max_tokens'] | Input should be less than or equal to 1024


## Step 6 — errors on purpose with HTTPException

Some inputs have the right *shape* but are still unacceptable (an empty message, a forbidden topic). Raise `HTTPException(status_code=400, detail=...)`. The client gets a clean error instead of a crash.

We rebuild `/chat` with this check. (Defining the same path again on the same app would register it twice, so we make a fresh app.)

In [7]:
from fastapi import HTTPException

app = FastAPI(title="Chat API v2")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    if not req.message.strip():
        raise HTTPException(status_code=400, detail="empty input")
    reply = fake_llm(req.message)
    return ChatResponse(reply=reply, input_tokens=rough_tokens(req.message), output_tokens=rough_tokens(reply))

client = TestClient(app)
print(client.post("/chat", json={"message": "   "}).json())
print(client.post("/chat", json={"message": "hello"}).status_code)

{'detail': 'empty input'}
200


## Step 7 — where the real LLM call goes (guarded)

This is the pattern used everywhere in the course: **call the real provider only if a key exists, otherwise fall back**. The key is read from an **environment variable** — never typed into the code. In this notebook there is no key, so you will see the offline path.

In [8]:
import os

def generate(prompt: str) -> str:
    if os.getenv("ANTHROPIC_API_KEY"):
        import anthropic                                      # pip install anthropic
        client_ = anthropic.Anthropic()                       # reads the key from the env var
        msg = client_.messages.create(
            model=os.getenv("LLM_MODEL", "claude-haiku-latest"),   # check the provider's model list
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        return msg.content[0].text
    return fake_llm(prompt)                                   # offline stand-in

print(generate("Say hello"))

(offline demo model) You asked about: Say hello


## Step 8 — async: why it matters, measured

An LLM call is mostly **waiting** for a remote server. Two kinds of waiting code:

- `time.sleep(0.5)` **blocks**: the whole program stops for 0.5 s.
- `await asyncio.sleep(0.5)` **pauses only this request**; the server serves others meanwhile.

We define three endpoints and fire **5 requests at the same time** at each, using `httpx.AsyncClient` with an `ASGITransport` (it talks to the app in-process, like TestClient, but concurrently).

Predict before running: which one takes ~0.5 s total and which takes ~2.5 s?

In [9]:
import asyncio, time, httpx

slow_app = FastAPI()

@slow_app.get("/async-good")
async def async_good():
    await asyncio.sleep(0.5)          # non-blocking wait (like awaiting an async LLM client)
    return {"ok": True}

@slow_app.get("/async-bad")
async def async_bad():
    time.sleep(0.5)                   # BLOCKING call inside async def: freezes everyone
    return {"ok": True}

@slow_app.get("/sync-def")
def sync_def():
    time.sleep(0.5)                   # blocking, but plain def -> FastAPI runs it in a worker thread
    return {"ok": True}

async def hit(path: str, n: int = 5) -> float:
    transport = httpx.ASGITransport(app=slow_app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        t0 = time.perf_counter()
        await asyncio.gather(*[ac.get(path) for _ in range(n)])   # send n requests at once
        return time.perf_counter() - t0

for path in ["/async-good", "/async-bad", "/sync-def"]:
    print(f"{path:12s} 5 concurrent requests took {await hit(path):.2f} s")

/async-good  5 concurrent requests took 0.50 s


/async-bad   5 concurrent requests took 2.51 s


/sync-def    5 concurrent requests took 0.51 s


**Read the result:**
- `/async-good` ≈ 0.5 s — all five waited *together*.
- `/async-bad` ≈ 2.5 s — each blocked the others: the classic beginner bug.
- `/sync-def` ≈ 0.5 s — FastAPI moved the blocking work to threads. So: if your library is blocking, use plain `def`; if it is async (`httpx.AsyncClient`, `anthropic.AsyncAnthropic`, `openai.AsyncOpenAI`), use `async def` + `await`.

(Exact times vary a little on a busy machine.)

## Step 9 — a simple API-key check with a dependency

Public APIs need to know *who* is calling. FastAPI's `Depends` runs a helper before your route. Here the helper reads a header called `X-API-Key` and compares it with a value from an environment variable. We set a harmless demo value in the notebook — in real life it comes from a secret store.

In [10]:
from fastapi import Depends, Header

os.environ.setdefault("SERVICE_API_KEY", "demo-only-not-secret")

def require_key(x_api_key: str = Header(default="")):
    if x_api_key != os.environ["SERVICE_API_KEY"]:
        raise HTTPException(status_code=401, detail="missing or wrong API key")   # 401 = not authenticated

@app.get("/private", dependencies=[Depends(require_key)])
def private():
    return {"secret_data": 42}

print(client.get("/private").status_code)
print(client.get("/private", headers={"X-API-Key": "demo-only-not-secret"}).json())

401
{'secret_data': 42}


## Step 10 — automatic documentation

FastAPI writes an **OpenAPI** description (a standard machine-readable description of an API) for you. When the server runs, `/docs` shows it as a clickable page. Here we peek at the raw description.

In [11]:
spec = app.openapi()
for path, methods in spec["paths"].items():
    print(path, "->", list(methods))
print("\nChatRequest fields:", list(spec["components"]["schemas"]["ChatRequest"]["properties"]))

/health -> ['get']
/chat -> ['post']
/private -> ['get']

ChatRequest fields: ['message', 'max_tokens']


## Step 11 — write tests like a professional

A **test** is a function whose name starts with `test_` and that uses `assert` ("this must be true"). **pytest** finds and runs them; GitHub Actions runs them on every push (see notebook 04 and `code/ci/`). Here we call them by hand.

In [12]:
def test_health():
    assert client.get("/health").json() == {"status": "ok"}

def test_chat_ok():
    r = client.post("/chat", json={"message": "hi"})
    assert r.status_code == 200 and r.json()["input_tokens"] >= 1

def test_chat_empty():
    assert client.post("/chat", json={"message": ""}).status_code == 400

def test_private_needs_key():
    assert client.get("/private").status_code == 401

for t in [test_health, test_chat_ok, test_chat_empty, test_private_needs_key]:
    t()
    print("PASS", t.__name__)

PASS test_health
PASS test_chat_ok
PASS test_chat_empty
PASS test_private_needs_key


## Step 12 — run a real server with uvicorn

So far nothing listened on the network. Now we start **uvicorn** (the server program) in a background **thread** on a free **port** (a numbered door on the computer), call it with a real HTTP request, then stop it.

On your own machine you would run in a terminal: `uvicorn app.main:app --reload --port 8000` and open `http://127.0.0.1:8000/docs`.

In [13]:
import socket, threading, uvicorn

with socket.socket() as s:                 # ask the OS for a free port
    s.bind(("127.0.0.1", 0)); port = s.getsockname()[1]

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
for _ in range(50):                        # wait up to 5 s for start-up
    if server.started: break
    time.sleep(0.1)

print(httpx.get(f"http://127.0.0.1:{port}/health").json())
print(httpx.post(f"http://127.0.0.1:{port}/chat", json={"message": "real HTTP!"}).json()["reply"])
server.should_exit = True                  # ask the server to stop
thread.join(timeout=5)
print("server stopped:", not thread.is_alive())

{'status': 'ok'}


(offline demo model) You asked about: real HTTP!
server stopped: True


## Recap

- A route = method + path + Python function. Type hints + pydantic give you validation and docs for free.
- 4xx = caller's fault (400 bad input, 401 no key, 404 no such path, 422 wrong shape); 5xx = ours.
- Use `async def` only with async libraries; blocking code goes in plain `def`.
- TestClient makes API tests fast enough to run on every change.

**Next:** open `code/app/` to see the same ideas organised as a real project, then notebook 02 to package it with Docker.

**Try it yourself:** add a `GET /tokens?text=...` route that returns the rough token count, and a test for it.